<a href="https://colab.research.google.com/github/kosmasrio0411/Final-KSSC-RAG/blob/main/RAG_Qwen2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 RAG Saturation & Hallucination Analysis
## Kapita Selekta SC — Final Assignment
### Dataset: IDK-MRC | Model: Qwen2.5-1.5B-Instruct (4-bit) | Retriever: FAISS

**Deskripsi:**  
Notebook ini mengimplementasikan sistem RAG lengkap untuk menginvestigasi:
1. **Saturasi performa** — apakah kualitas jawaban terus meningkat seiring bertambahnya K?
2. **Halusinasi sitasi** — seberapa sering LLM mengutip dokumen yang salah?

**Eksperimen:** K ∈ {1, 3, 5, 10} diuji menggunakan metrik ROUGE, BLEU, dan BERTScore.

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install semua dependensi yang dibutuhkan
# Runtime: ~2-3 menit pada Google Colab
# ============================================================
!pip install -q datasets faiss-cpu sentence-transformers
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q rouge-score bert-score nltk
!pip install -q matplotlib seaborn pandas tqdm openpyxl


import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Semua dependensi berhasil diinstall!")


## 📦 Cell 2 — Import Libraries

In [ ]:
# ============================================================
# CELL 2: Import semua library yang diperlukan
# ============================================================
import os
import gc
import re
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

import torch
print(f"🖥️  PyTorch version : {torch.__version__}")
print(f"🖥️  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"🖥️  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from datasets import load_dataset
import faiss
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("\n✅ Semua library berhasil diimport!")


## 📂 Cell 3 — Load & Eksplorasi Dataset IDK-MRC

In [ ]:
# ============================================================
# CELL 3: Load dataset IDK-MRC dari HuggingFace
# IDK-MRC = Indonesian Knowledge Machine Reading Comprehension
# Format: setiap item memiliki 'context' dan 'qas' (daftar QA)
# ============================================================
print("⏳ Memuat dataset IDK-MRC...")
ds = load_dataset("rifkiaputri/idk-mrc")
print("\n✅ Dataset berhasil dimuat!")
print(f"\n{ds}")

# --- Eksplorasi struktur data ---
print("\n" + "="*60)
print("STRUKTUR DATA (baris pertama split 'train')")
print("="*60)
sample = ds['train'][0]
print(f"\n📝 Context (200 karakter pertama):")
print(f"  {sample['context'][:200]}...")
print(f"\n❓ Jumlah QA pairs pada baris ini: {len(sample['qas'])}")
for i, qa in enumerate(sample['qas']):
    print(f"\n  QA-{i+1}:")
    print(f"    Question   : {qa['question']}")
    print(f"    Is Impossible: {qa['is_impossible']}")
    if qa['answers']:
        print(f"    Answer     : {qa['answers'][0]['text']}")
    else:
        print(f"    Answer     : (tidak ada — pertanyaan tidak bisa dijawab)")

# --- Statistik split ---
print("\n" + "="*60)
print("STATISTIK DATASET")
print("="*60)
for split_name in ['train', 'validation', 'test']:
    split = ds[split_name]
    total_qa = sum(len(row['qas']) for row in split)
    answerable = sum(
        sum(1 for qa in row['qas'] if not qa['is_impossible'])
        for row in split
    )
    print(f"  {split_name:12s}: {len(split):5d} konteks | "
          f"{total_qa:5d} total QA | {answerable:5d} bisa dijawab")


## 🔧 Cell 4 — Preprocessing & Persiapan Data

In [ ]:
# ============================================================
# CELL 4: Preprocessing data
# Kita ekstrak QA pairs yang bisa dijawab (is_impossible=False)
# dari split validation untuk evaluasi.
# ============================================================

def extract_qa_pairs(dataset_split, include_impossible=False):
    """
    Ekstrak QA pairs dari satu split dataset.

    Returns:
        contexts (list): konteks per QA pair
        questions (list): pertanyaan
        answers (list): jawaban ground truth
        context_idx (list): indeks baris asli dalam split
    """
    contexts, questions, answers, ctx_indices = [], [], [], []

    for row_idx, row in enumerate(dataset_split):
        ctx = row['context']
        for qa in row['qas']:
            if qa['is_impossible']:
                if include_impossible:
                    contexts.append(ctx)
                    questions.append(qa['question'])
                    answers.append("")  # tidak ada jawaban
                    ctx_indices.append(row_idx)
                continue
            if not qa['answers']:
                continue
            contexts.append(ctx)
            questions.append(qa['question'])
            answers.append(qa['answers'][0]['text'])
            ctx_indices.append(row_idx)

    return contexts, questions, answers, ctx_indices


# --- Ekstrak dari validation split ---
val_gt_contexts, val_questions, val_answers, val_row_indices =     extract_qa_pairs(ds['validation'], include_impossible=False)

print(f"✅ Total QA pairs bisa dijawab (validation): {len(val_questions)}")
print(f"\nContoh QA pair:")
print(f"  Pertanyaan : {val_questions[0]}")
print(f"  Jawaban    : {val_answers[0]}")
print(f"  Konteks    : {val_gt_contexts[0][:120]}...")

# --- Juga ekstrak train untuk building corpus yang lebih besar ---
train_gt_contexts, train_questions, train_answers, train_row_indices =     extract_qa_pairs(ds['train'], include_impossible=False)

print(f"\n✅ Total QA pairs bisa dijawab (train): {len(train_questions)}")

# --- Statistik panjang teks ---
ctx_lengths  = [len(c.split()) for c in val_gt_contexts[:500]]
q_lengths    = [len(q.split()) for q in val_questions[:500]]
ans_lengths  = [len(a.split()) for a in val_answers[:500]]

print("\n" + "="*60)
print("STATISTIK PANJANG TEKS (validation)")
print("="*60)
print(f"  Konteks  — rata-rata: {np.mean(ctx_lengths):.1f} kata | "
      f"max: {max(ctx_lengths)} kata")
print(f"  Pertanyaan — rata-rata: {np.mean(q_lengths):.1f} kata | "
      f"max: {max(q_lengths)} kata")
print(f"  Jawaban  — rata-rata: {np.mean(ans_lengths):.1f} kata | "
      f"max: {max(ans_lengths)} kata")


## 🗄️ Cell 5 — Bangun Corpus Retrieval

In [ ]:
# ============================================================
# CELL 5: Bangun corpus untuk retrieval
# Corpus = semua konteks unik dari train + validation + test
# Ini mensimulasikan knowledge base yang sesungguhnya
# ============================================================

print("⏳ Membangun corpus dari seluruh split...")

all_corpus_contexts = []
seen = set()

for split_name in ['train', 'validation', 'test']:
    for row in ds[split_name]:
        ctx = row['context'].strip()
        if ctx not in seen:
            seen.add(ctx)
            all_corpus_contexts.append(ctx)

print(f"\n✅ Corpus siap!")
print(f"   Total konteks unik : {len(all_corpus_contexts)}")

# Buat mapping konteks → indeks untuk pencarian cepat
corpus_ctx_to_idx = {ctx: i for i, ctx in enumerate(all_corpus_contexts)}

# Verifikasi: pastikan semua GT contexts ada dalam corpus
missing = sum(
    1 for ctx in val_gt_contexts
    if ctx not in corpus_ctx_to_idx
)
print(f"   GT contexts tidak ditemukan di corpus: {missing} "
      f"(harusnya 0 atau sangat kecil)")

# Contoh corpus
print(f"\nContoh konteks korpus ke-0:")
print(f"  {all_corpus_contexts[0][:200]}...")



## 🧠 Cell 6 — Load Embedding Model (Sentence Transformer)

In [ ]:
# ============================================================
# CELL 6: Load embedding model
# paraphrase-multilingual-MiniLM-L12-v2:
#   - Mendukung 50+ bahasa termasuk Bahasa Indonesia
#   - Ringan (~420 MB) dan cepat
#   - Dimensi embedding: 384
# ============================================================

EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Embedding model dimuat!")
print(f"   Dimensi output       : {embed_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length  : {embed_model.max_seq_length}")

# Test embedding
test_sentence = "Dimana Douwes Dekker dilahirkan?"
test_emb = embed_model.encode([test_sentence])
print(f"\nTest embedding:")
print(f"  Input   : '{test_sentence}'")
print(f"  Shape   : {test_emb.shape}")
print(f"  Norm    : {np.linalg.norm(test_emb):.4f}")


## 🔎 Cell 7 — Bangun FAISS Index

In [ ]:
# ============================================================
# CELL 7: Encode semua corpus contexts & bangun FAISS index
# Menggunakan IndexFlatIP (Inner Product) dengan normalisasi L2
# → efektif = cosine similarity
#
# Estimasi waktu: ~3-5 menit untuk ~4000 konteks di Colab
# ============================================================

EMBED_DIM  = embed_model.get_sentence_embedding_dimension()
BATCH_SIZE = 64

print(f"⏳ Encoding {len(all_corpus_contexts)} konteks...")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Dimensi    : {EMBED_DIM}")

corpus_embeddings_list = []
for i in tqdm(range(0, len(all_corpus_contexts), BATCH_SIZE),
              desc="Encoding corpus"):
    batch = all_corpus_contexts[i : i + BATCH_SIZE]
    embs  = embed_model.encode(batch,
                               convert_to_numpy=True,
                               show_progress_bar=False,
                               normalize_embeddings=True)  # L2-norm in-place
    corpus_embeddings_list.append(embs)

corpus_embeddings = np.vstack(corpus_embeddings_list).astype('float32')
print(f"\n✅ Encoding selesai! Shape: {corpus_embeddings.shape}")

# --- Bangun FAISS index ---
print("\n⏳ Membangun FAISS index (IndexFlatIP)...")
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(corpus_embeddings)

print(f"✅ FAISS index siap!")
print(f"   Total vektor tersimpan : {faiss_index.ntotal}")
print(f"   Tipe index             : {type(faiss_index).__name__}")

# --- Free memory ---
del corpus_embeddings_list
gc.collect()
